# Residual-diffusion precipitation nowcasting: demo

This notebook loads the trained models of the project and compares, on a few radar events from the test split:
persistence, optical-flow advection, two U-Nets (MSE and weighted MSE) and a residual diffusion model.
Given 13 radar frames (5 minutes apart, about one hour), each model forecasts the next 12 frames (+5 to +60 min).

**What to expect.** The U-Nets are accurate but blurry, advection keeps the shapes but is only a translation,
and the diffusion model produces realistic detail that is *not* more accurate cell by cell (see the README for the
measured trade-offs). The last cells show the diffusion model as an ensemble.

**Runtime.** A GPU is strongly recommended (Colab: *Runtime > Change runtime type > GPU*). On CPU the notebook
works but the diffusion sampling is slow.

In [11]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/bamont/nowcasting-weather.git"


def find_root():
    """Repository root: the first folder (from here upwards) that contains scripts/make_gifs.py."""
    return next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "make_gifs.py").exists()), None)


root = find_root()
if root is None:                      # not inside the repository (e.g. Colab): clone it
    if "<" in REPO_URL:
        raise RuntimeError("Set REPO_URL to your repository, or open this notebook inside a local clone.")
    !git clone -q {REPO_URL}
    root = Path("nowcasting-weather").resolve()
%cd {root}
sys.path.insert(0, str(root / "src"))   # `import nowcasting` works even if the package was installed after the kernel started

try:
    import pytorch_lightning, cv2, h5py  # noqa: F401  (dependencies of the project)
except ImportError:
    !pip install -q -e . 2>&1 | tail -n 2
print("repository:", root)

c:\Users\baptm\Downloads\nowcasting
repository: c:\Users\baptm\Downloads\nowcasting


## 1. Download the model weights and the demo samples

The files are release assets of the repository. Set `RELEASE_BASE` to the download URL of the release, or upload the four files
to the `demo_files/` folder yourself. `demo_samples.npz` is created with `scripts/make_demo_data.py`.

In [12]:
from pathlib import Path
import urllib.request

RELEASE_BASE = "https://github.com/bamont/nowcasting-weather/releases/download/v1.0"
DEMO_DIR = Path("demo_files")
DEMO_DIR.mkdir(exist_ok=True)
FILES = ["unet_mse_best.ckpt", "unet_weighted_mse_best.ckpt", "diffusion_final.ckpt", "demo_samples.npz"]


def fetch(name):
    dest = DEMO_DIR / name
    if dest.exists():
        return dest
    if "<" in RELEASE_BASE:
        raise RuntimeError(f"{name} is missing: set RELEASE_BASE above, or upload the file to {DEMO_DIR}/")
    print("downloading", name)
    urllib.request.urlretrieve(f"{RELEASE_BASE}/{name}", dest)
    return dest


paths = {name: fetch(name) for name in FILES}
print({name: f"{p.stat().st_size / 1e6:.1f} MB" for name, p in paths.items()})

downloading unet_mse_best.ckpt


URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get issuer certificate (_ssl.c:1010)>

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Image, display

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("No GPU found: the diffusion model will be slow (Runtime > Change runtime type > GPU).")

## 2. Pick an event

Each event has 49 frames. A window uses 25 consecutive frames starting at `OFFSET` (0, 12 or 24): the first 13 are the input,
the next 12 are the target. The first events of the file are the most intense storms of the test split, the last ones were drawn
at random: the intense cases are the visually striking ones, the random cases are more representative.

In [ ]:
data = np.load(paths["demo_samples.npz"])
frames, kind = data["frames"], data["kind"]          # (N, 49, 128, 128) uint8, 'intense' or 'random'
print(frames.shape, list(kind))

EVENT = 0      # 0 .. N-1
OFFSET = 12    # 0, 12 or 24

seq = torch.from_numpy(frames[EVENT, OFFSET:OFFSET + 25]).float().div(255).unsqueeze(1)   # (25, 1, H, W) in [0, 1]
x, y = seq[:13][None], seq[13:][None]                # 13 observed frames -> 12 future frames
print(f"event {EVENT} ({kind[EVENT]}), input {tuple(x.shape)}, target {tuple(y.shape)}")

## 3. Load the five forecasting models

Persistence and advection need no weights. Advection uses the settings tuned on the validation split.

In [ ]:
from nowcasting.models.loading import load_models

ADVECTION_LOOKBACK, ADVECTION_WINSIZE = 4, 31
models = load_models(
    [f"U-Net (MSE)={paths['unet_mse_best.ckpt']}", f"U-Net (weighted MSE)={paths['unet_weighted_mse_best.ckpt']}"],
    out_len=12,
    lookback=ADVECTION_LOOKBACK,
    winsize=ADVECTION_WINSIZE,
    diffusion_specs=[f"Residual diffusion={paths['diffusion_final.ckpt']}"],
    sample_steps=20,
)
models = {name: m.to(device).eval() for name, m in models.items()}
print(list(models))

## 4. Forecast and compare

Rows: observation and each model. Columns: the last input frame, then +15, +30, +45 and +60 minutes.
The colour bar marks the VIL thresholds used in the evaluation. The diffusion forecast is a single random sample.

In [ ]:
with torch.no_grad():
    preds = {name: m(x.to(device))[0, :, 0].float().cpu().contiguous().numpy() for name, m in models.items()}
obs = y[0, :, 0].numpy()                              # (12, H, W)

import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "make_gifs.py").exists()), None)
if root is None:
    raise RuntimeError("Repository not found: run the first cell of the notebook.")
sys.path.insert(0, str(root / "scripts"))
import make_gifs

%matplotlib inline
# make_gifs selects the non-interactive Agg backend, so restore inline plotting for the next cells

Path("demo_out").mkdir(exist_ok=True)
make_gifs.make_png(Path("demo_out/comparison.png"), x[0, -1, 0].numpy(), obs, preds)
display(Image(filename="demo_out/comparison.png"))

In [ ]:
# animated version: all 12 lead times
make_gifs.make_gif(Path("demo_out/comparison.gif"), obs, preds)
display(Image(filename="demo_out/comparison.gif"))

## 5. How much small-scale detail does each forecast keep?

Ratio of the forecast power spectrum to the observed one at +60 min, by spatial scale (`k` = cycles per image;
large `k` = small structures). A value of 1 means the same amount of detail as the observation; the U-Nets fall far below 1,
the diffusion model recovers part of the detail. This is one window, so the curves are noisy: the project README reports the
statistics over the whole validation split.

In [ ]:
from nowcasting.sharpness import SpectrumAccumulator


def spectrum(arr):
    acc = SpectrumAccumulator(12, arr.shape[-1])
    acc.update(arr[None])
    return acc.compute()


T = 11  # +60 min
obs_spec = spectrum(obs)[T]
k = np.arange(len(obs_spec))
fig, ax = plt.subplots(figsize=(6.5, 4))
for name, arr in preds.items():
    ax.plot(k[1:], (spectrum(arr)[T] / obs_spec)[1:], label=name)
ax.axhline(1.0, color="k", ls="--", lw=0.8)
ax.set(xscale="log", yscale="log", xlabel="wavenumber k (cycles per image)",
       ylabel="power: forecast / observation", title=f"+{5 * (T + 1)} min")
ax.grid(alpha=0.3, which="both")
ax.legend(fontsize=8)
plt.show()

## 6. The diffusion model as an ensemble

Each sample starts from different noise. The members share the same large-scale forecast but differ in detail, and the fraction of members
above a threshold gives a probability map. In the evaluation (README) this ensemble is under-dispersive (spread/skill about 0.54):
its members are too similar for the error of the ensemble mean, so treat the probabilities as indicative.

In [ ]:
N_MEMBERS = 8 if device == "cuda" else 3
THRESHOLD = 133                                       # VIL level of the probability map (0-255 scale)
T = 11                                                # +60 min

lit = models["Residual diffusion"].lit
gen = torch.Generator(device=device)
gen.manual_seed(0)
with torch.no_grad():
    members = lit.sample(x.to(device), n_members=N_MEMBERS, steps=20, generator=gen)[0, :, :, 0].cpu().numpy()  # (M, 12, H, W)

panels = [("Observation", obs[T]), ("U-Net (MSE)", preds["U-Net (MSE)"][T])]
panels += [(f"Member {i + 1}", members[i, T]) for i in range(3)]
panels += [(f"Mean of {N_MEMBERS}", members[:, T].mean(axis=0))]
prob = (members[:, T] * 255 >= THRESHOLD).mean(axis=0)

fig, axes = plt.subplots(1, len(panels) + 1, figsize=(2.6 * (len(panels) + 1), 3.3))
for ax, (name, img) in zip(axes, panels):
    ax.imshow(img, vmin=0, vmax=1, cmap="turbo")
    ax.set_title(name, fontsize=9)
    ax.axis("off")
im = axes[-1].imshow(prob, vmin=0, vmax=1, cmap="Blues")
axes[-1].set_title(f"P(VIL >= {THRESHOLD})", fontsize=9)
axes[-1].axis("off")
fig.colorbar(im, ax=axes[-1], fraction=0.046)
fig.suptitle(f"+{5 * (T + 1)} min")
plt.show()

## 7. Scores on all the demo windows (GPU only)

Illustrative only: the file holds a handful of events (selected partly for their intensity), so these numbers are **not** the results of the project.
The reference results are the validation and test tables of the README. CSI-M is the mean CSI over six VIL thresholds, the frequency bias is the
forecast area above a threshold divided by the observed area (1 is ideal).

In [ ]:
from nowcasting.metrics import NowcastingMetrics, THRESHOLDS

if device != "cuda":
    print("Skipped on CPU: the diffusion model would take too long.")
else:
    metrics = {name: NowcastingMetrics(12) for name in models}
    with torch.no_grad():
        for e in range(frames.shape[0]):
            for offset in (0, 12, 24):
                s = torch.from_numpy(frames[e, offset:offset + 25]).float().div(255).unsqueeze(1)
                xx, yy = s[:13][None].to(device), s[13:][None]
                for name, m in models.items():
                    metrics[name].update(m(xx), yy)
    i181 = THRESHOLDS.index(181)
    print(f"{frames.shape[0] * 3} windows")
    print(f"{'model':24s} {'CSI-M':>6s} {'bias@181':>9s} {'MSE':>8s}")
    for name, m in metrics.items():
        res = m.compute()
        print(f"{name:24s} {float(torch.nanmean(res['csi_pooled'])):6.3f} {float(m.frequency_bias()[i181]):9.2f} {float(res['mse'].mean()):8.5f}")

## Going further

- Change `EVENT` and `OFFSET` in section 2 and re-run from there.
- Use your own data: any array of shape `(49, 128, 128)` with VIL values 0-255 (5 minutes apart, 3 km per pixel) works the same way.
- Evaluation protocol, measured results, limitations and the experiments that did not help are in the repository README.